In [3]:
import joblib
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier


X_train = joblib.load('data/X_train.pkl')
X_test = joblib.load('data/X_test.pkl')
y_train = joblib.load('data/y_train.pkl')
y_test = joblib.load('data/y_test.pkl')

I engineered 5 domain informed features from the orbital elements.perihelion proximity to Earth's orbit, aphelion proximity to Earth's orbit, orbit spread, an Earth-crossing flag, and an eccentricity-inclination combined factor 

In [4]:
for data in [X_train, X_test]:

    data['q_proximity_1au'] = abs(data['q'] - 1.0)
    
    data['ad_proximity_1au'] = abs(data['ad'] - 1.0)
    
    data['orbit_spread'] = data['ad'] - data['q']
    
    data['earth_crossing'] = ((data['q'] < 1.0) & (data['ad'] > 1.0)).astype(int)
    
    data['ecc_incl_factor'] = data['e'] / (1 + np.radians(data['i']))

In [5]:
new_features = ['q_proximity_1au', 'ad_proximity_1au', 'orbit_spread', 'earth_crossing', 'ecc_incl_factor']
existing_features = ['e', 'a', 'i', 'q', 'ad', 'per_y', 'n']

corr_matrix = X_train[new_features + existing_features].corr()
corr_matrix.loc[new_features, existing_features]

,e,a,i,q,ad,per_y,n
q_proximity_1au,0.384790,-0.086716,0.043520,-0.959501,-0.038915,-0.013883,0.437198
ad_proximity_1au,-0.165873,0.879618,0.112941,-0.029201,0.884628,0.947362,0.188290
orbit_spread,0.422836,0.650097,0.069785,-0.687209,0.687209,0.681383,0.172927
earth_crossing,0.058806,0.104248,0.066032,-0.005303,0.104939,0.041480,-0.038938
ecc_incl_factor,0.999818,0.200325,-0.053691,-0.355873,0.219000,0.042182,-0.587671


In [6]:
X_train_with_target = X_train.copy()
X_train_with_target['pha_binary'] = y_train.values

target_corr = X_train_with_target.corr()['pha_binary'].sort_values(ascending=False)
print(target_corr)

pha_binary          1.000000
data_arc            0.265247
data_arc_years      0.265244
q_proximity_1au     0.164111
ecc_incl_factor     0.153804
e                   0.153577
class_APO           0.148928
orbit_spread        0.135876
i                   0.061688
class_IEO           0.023817
ad                  0.015508
earth_crossing      0.014191
a                   0.006889
per                -0.000695
per_y              -0.000697
class_ATE          -0.004606
n                  -0.017515
ad_proximity_1au   -0.018602
class_AMO          -0.153743
q                  -0.171242
H                  -0.306646
condition_code     -0.312185
Name: pha_binary, dtype: float64


In [7]:
rf_check = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_check.fit(X_train, y_train)

importances = pd.Series(rf_check.feature_importances_, index=X_train.columns)
importances.sort_values(ascending=False)

H                   0.348353
data_arc            0.116601
data_arc_years      0.085979
q                   0.080825
condition_code      0.064780
orbit_spread        0.057168
q_proximity_1au     0.037281
i                   0.030700
e                   0.028486
ecc_incl_factor     0.027161
class_AMO           0.022226
class_APO           0.015212
a                   0.015077
n                   0.014847
ad                  0.014269
per                 0.014081
ad_proximity_1au    0.013721
per_y               0.012708
class_ATE           0.000449
class_IEO           0.000041
earth_crossing      0.000035
dtype: float64

In [8]:
to_drop = ['ecc_incl_factor', 'ad_proximity_1au', 'earth_crossing']

X_train = X_train.drop(columns=to_drop)
X_test = X_test.drop(columns=to_drop)

In [9]:
joblib.dump(X_train, 'data/X_train_fe.pkl')
joblib.dump(X_test, 'data/X_test_fe.pkl')

['data/X_test_fe.pkl']

In [10]:
joblib.dump(X_train.columns.tolist(), 'data/feature_columns.pkl')

['data/feature_columns.pkl']

In [ ]:
feature_columns = joblib.load('data/feature_columns.pkl')
print(feature_columns)

['H', 'e', 'a', 'i', 'q', 'ad', 'per', 'per_y', 'n', 'condition_code', 'data_arc', 'data_arc_years', 'class_AMO', 'class_APO', 'class_ATE', 'class_IEO', 'q_proximity_1au', 'orbit_spread']


In [15]:
scaler = joblib.load('data/scaler.pkl')
print(scaler.feature_names_in_)

['H' 'e' 'a' 'i' 'q' 'ad' 'per' 'per_y' 'n' 'condition_code' 'data_arc'
 'data_arc_years']
